# TREC22 one-shot — frozen v7 monoT5_CT on the NQS pool (the real SOTA test)

Everything in §15 is **TREC21 develop**. This is the untouched **TREC22** test. The recipe (λ_KL=1,
KZ+synth, LoRA r8/α8 q-v) TOUCH #2: the `opus.v3_implicit` recipe (§17), frozen at **step 2500** (dev 0.6617 — dev-max,
and canary held 99% so there is no overfit reason to freeze earlier). The v1 touch #1 gave 0.5570.

**Anti-gaming discipline (load-bearing):** this notebook evaluates **exactly one** checkpoint on TREC22,
**once**. Do **not** point `ADAPTER` at several checkpoints and keep the best — that is tuning on the test
and destroys the one-shot's validity. Pick the develop-selected checkpoint, run, read the number, stop.

**What it does:** reranks the **same TREC22 NQS pool** the frozen pipeline uses (`pool_nqs.json`), scoring
each (topic, pool-doc) with base monoT5-3B-MED + the frozen v7 LoRA adapter, in the **exact doc format the
model was trained/dev-eval'd on** (`title: … condition: … eligibility: …`[:1400], true/false margin), and
computes graded **NDCG@10** with `pytrec_eval` — the identical metric that gives the baselines below.

**How to read the number (interpretation guardrails):**
- **The fair bar is 0.5750 — our ensemble on this *same NQS pool* — not the 0.6132 develop number.** Develop was the TREC21 *judged* pool (all judged docs, easy); this is the TREC22 *NQS* pool (retrieval-limited). A result below 0.613 is *expected* and is mostly the pool-type change, **not** a generalization failure.
- **Clean control = a zero-shot pass in the *same vdoc format + pool*** (base monoT5, adapter off). That treatment−control delta is "what the synth adaptation actually added on the test." (The §13h 0.5220 zero-shot used a *different* doc format — elig_first — so it's a footnote, not the control.)
- **h2oloo 0.6125 is FULL-CORPUS.** Beating it on our NQS pool is *competitive, pool-caveated* — not a clean SOTA beat. A clean beat needs full-corpus retrieve→rerank (or h2oloo's TREC22 run file for a paired test).
- Pool oracle is 0.957 (§11c), so the pool can support far higher than any of these.

In [ ]:
!pip install -q git+https://github.com/semajyllek/ctmatch.git
!pip install -q transformers accelerate sentencepiece peft pytrec_eval datasets tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json
os.environ['HF_HUB_DISABLE_XET'] = '1'; os.environ['HF_HOME'] = '/content/hf_cache'
import numpy as np, torch, pytrec_eval
from tqdm.auto import tqdm

DATA_ROOT  = '/content/drive/MyDrive/ct_data23'
BASE_MODEL = 'castorini/monot5-3b-med-msmarco'
# The ONE checkpoint frozen on TREC21 develop. TOUCH #2 = opus.v3_implicit recipe, step 2500
# (dev 0.6617 — dev-max AND canary 99%, no forgetting, so dev-max is the right freeze; §17d).
# ⚠ the Opus run reused the monot5_ct_v7_step* paths, so these are the OPUS checkpoints now.
# PICK ONE — do not run several on TREC22 (that turns the test into a dev set).
ADAPTER    = f'{DATA_ROOT}/monot5_ct_v7_step2500'
DOC_CHARS  = 1400                       # the char cap the v7 model was trained/eval'd with
device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert os.path.exists(ADAPTER), f'adapter not found: {ADAPTER} (run the v7 STEPS>=1250 training first)'
print('one-shot adapter:', ADAPTER)

In [ ]:
# ── Load the TREC22 NQS pool + qrels + corpus fields (mirrors exp_reranker_bakeoff for comparability) ──
from ctmatch.experiments import ExperimentConfig, load_corpus, load_eval
cfg = ExperimentConfig(data_root=DATA_ROOT, pool_tag='nqs')
corpus_ids, corpus_fields = load_corpus(cfg); id2f = dict(zip(corpus_ids, corpus_fields))
sets = load_eval(cfg, ['trec22'])
t2t22 = sets['trec22']['topic2text']; rel22 = sets['trec22']['rel_dict']
pool = json.load(open(cfg.path('data/pool_nqs.json')))
pool22 = pool['trec22']
n_pairs = sum(len(v) for v in pool22.values())
print(f'TREC22 NQS pool: {len(pool22)} topics, {n_pairs:,} (topic, doc) pairs to rerank')

def ndcg22(scores):
    run, qr = {}, {}
    for t in pool22:
        docs = [d for d in pool22[t] if (t, d) in scores]
        if not docs or t not in rel22: continue
        run[t] = {d: scores[(t, d)] for d in docs}
        qr[t]  = {d: int(r) for d, r in rel22[t].items()}   # graded qrels (rel 0/1/2)
    ev = pytrec_eval.RelevanceEvaluator(qr, {'ndcg_cut.10'}).evaluate(run)
    return float(np.mean([v['ndcg_cut_10'] for v in ev.values()]))

In [ ]:
# ── Load base monoT5 + frozen v7 adapter; score the pool (v7 + a same-format zero-shot control) ──
from transformers import T5Tokenizer, T5ForConditionalGeneration
from peft import PeftModel
tok = T5Tokenizer.from_pretrained(BASE_MODEL)
base = T5ForConditionalGeneration.from_pretrained(BASE_MODEL, torch_dtype=torch.float16).to(device).eval()
model = PeftModel.from_pretrained(base, ADAPTER).eval()
TRUE = tok('true', add_special_tokens=False).input_ids[0]; FALSE = tok('false', add_special_tokens=False).input_ids[0]

def vdoc(df):                                   # EXACT format the v7 model was trained + dev-eval'd on
    t = df.get('brief_title') or df.get('official_title') or ''
    c = df.get('conditions', ''); c = ', '.join(str(x) for x in c if x) if isinstance(c, list) else (c or '')
    e = df.get('eligibility', '') or ''
    return f'title: {t} condition: {c} eligibility: {e}'[:DOC_CHARS]

@torch.no_grad()
def mt_score(q, docs, adapter=True, b=16):
    out = []
    for i in range(0, len(docs), b):
        ins = [f'Query: {q} Document: {d} Relevant:' for d in docs[i:i+b]]
        enc = tok(ins, return_tensors='pt', padding=True, truncation=True, max_length=512).to(device)
        dec = torch.zeros((enc['input_ids'].shape[0], 1), dtype=torch.long, device=device)
        if adapter:
            lg = model(**enc, decoder_input_ids=dec).logits[:, 0, :].float()
        else:                                    # zero-shot control: same base weights, adapter OFF
            with model.disable_adapter():
                lg = model(**enc, decoder_input_ids=dec).logits[:, 0, :].float()
        lp = torch.log_softmax(lg, -1)
        out.extend((lp[:, TRUE] - lp[:, FALSE]).cpu().tolist())
    return out

def rerank_pool(tag, adapter):                   # resumable; one cache file per (tag)
    cache = cfg.path(f'data/oneshot_{tag}_trec22.jsonl'); sc, done = {}, set()
    if os.path.exists(cache):
        for l in open(cache):
            r = json.loads(l); sc[(r['t'], r['d'])] = r['s']; done.add(r['t'])
    todo = [t for t in pool22 if t in t2t22 and t not in done]
    with open(cache, 'a') as f:
        for t in tqdm(todo, desc=f'rerank {tag}'):
            docs = [d for d in pool22[t] if d in id2f]
            if not docs: continue
            for d, s in zip(docs, mt_score(t2t22[t], [vdoc(id2f[d]) for d in docs], adapter=adapter)):
                sc[(t, d)] = s; f.write(json.dumps({'t': t, 'd': d, 's': float(s)}) + '\n')
            f.flush()
    return sc

r22   = ndcg22(rerank_pool(os.path.basename(ADAPTER), adapter=True))   # v7 (synth-adapted, treatment)
rbase = ndcg22(rerank_pool('base_vdoc', adapter=False))               # zero-shot control (same vdoc + pool)

print('\n' + '=' * 66)
print(f'  TREC22 ONE-SHOT — frozen {os.path.basename(ADAPTER)} on the NQS pool')
print('=' * 66)
print(f'  v7 monoT5_CT (synth-adapted)       NDCG@10 = {r22:.4f}')
print(f'  zero-shot control (base, same vdoc+pool)    {rbase:.4f}   <- adaptation added {r22 - rbase:+.4f}')
print('  --- same NQS pool ---')
print('  our ensemble (R/multi-view/NQS, §12)        0.5750   <- the fair internal bar')
print('  monoT5-MED zero-shot, elig_first (§13h)     0.5220   (different doc format; footnote)')
print('  --- SOTA reference (h2oloo, FULL-CORPUS) ---')
print('  h2oloo frocchio_monot5_e (TREC22 winner)    0.6125   (full-corpus retrieval != our pool)')
print('=' * 66)
if r22 > 0.5750:
    print(f'  => beats the fair ensemble bar (0.575); synth adaptation delta vs control = {r22 - rbase:+.4f}.')
elif r22 > rbase:
    print(f'  => adaptation helped on TREC22 ({r22 - rbase:+.4f} over the control) but below the 0.575 bar.')
else:
    print('  => no gain over the zero-shot control on TREC22 — the develop gains did not transfer.')
if r22 > 0.6125:
    print('  => also clears h2oloo 0.6125 — COMPETITIVE, POOL-CAVEATED (our NQS pool vs their full-corpus), not a clean beat.')
json.dump({'adapter': os.path.basename(ADAPTER), 'trec22_nqs_ndcg10': round(r22, 4),
           'zeroshot_control_vdoc': round(rbase, 4), 'adaptation_delta': round(r22 - rbase, 4)},
          open(cfg.path('data/trec22_oneshot_result.json'), 'w'))
print('result ->', cfg.path('data/trec22_oneshot_result.json'))